# Kelompok 06 · Training di Google Colab

Sebelum run:
1. Upload `train.zip` ke Google Drive, folder `MyDrive/sampah-studio2/`.
2. **Runtime → Change runtime type → T4 GPU**.

`model.pth` dan `experiments/` otomatis disalin ke `MyDrive/sampah-studio2/hasil/`.

In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/sampah-studio2'  # sesuaikan kalau beda
!ls -lh {DRIVE_DIR}

In [ ]:
# copy dataset ke disk Colab dulu (baca langsung dari Drive lambat)
!cp {DRIVE_DIR}/train.zip /content/train.zip
!unzip -q -o /content/train.zip -d /content/data
DATA = '/content/data/train'
!for d in {DATA}/*/; do echo "$d $(ls $d | wc -l)"; done

In [ ]:
!git clone https://github.com/beningapniprameswari/Studio-2_Kelompok-6.git /content/Studio-2_Kelompok-6
%cd /content/Studio-2_Kelompok-6
!pip install -q -r submission/requirements.txt
!mkdir -p experiments
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline ResNet-18 (opsional)
Bobot masuk ke `baseline/model.pth`, submission utama tidak ketimpa.

In [ ]:
!python train_baseline.py --data {DATA} --epochs 8 2>&1 | tee experiments/01_baseline_resnet18.txt
!python evaluate.py --submission ./baseline --data ./my_val 2>&1 | tee -a experiments/01_baseline_resnet18.txt

## 2. ConvNeXt-Tiny
Train di 85% data, epoch terbaik dipilih dari Macro-F1 di `my_val/`.

In [ ]:
!python train.py --data {DATA} --epochs 15 --workers 2 2>&1 | tee experiments/02_convnext_tiny.txt

In [ ]:
# cek submission pakai evaluate.py, lalu simpan hasil ke Drive
!python evaluate.py --submission ./submission --data ./my_val 2>&1 | tee -a experiments/02_convnext_tiny.txt
!ls -lh submission/model.pth
!mkdir -p {DRIVE_DIR}/hasil
!cp submission/model.pth {DRIVE_DIR}/hasil/model.pth
!cp -r experiments {DRIVE_DIR}/hasil/
!ls -lhR {DRIVE_DIR}/hasil